# Multi-session dimensionality at a matched population of 50 neurons

Every quantity here is measured on the same object: the eigenvalue spectrum of a
**50-neuron subsample** of one population in one brain state, averaged over 50
random subsamples. Three summaries of that spectrum are compared:

| symbol | definition | needs a fitted range? |
|---|---|---|
| $D_{\mathrm{PR}}$ | participation ratio, $(\sum\lambda)^2 / \sum\lambda^2$ | no |
| $D_{90}$ | components needed for 90 % of the variance | no |
| $\hat{\alpha}$ | Clauset maximum-likelihood tail index above a KS-selected cutoff | no (cutoff is data-selected) |

The exponent is reported **by maximum likelihood**, not least squares, and is
converted to rank units by $\alpha = 1/(\hat{\alpha}-1)$ so it can be read
against the earlier single-session figures.

Spectra come from the **covariance** of mean-centred spike counts; counts are
not z-scored, so neurons with more variance contribute more (see Methods).


In [ ]:
import os, sys, json, hashlib, warnings
import numpy as np, pandas as pd
import matplotlib as mpl, matplotlib.pyplot as plt
import powerlaw
from scipy.stats import spearmanr, wilcoxon
warnings.filterwarnings("ignore")

HERE = os.path.dirname(os.path.abspath("__file__")) if "__file__" not in globals() else os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, os.path.abspath("../scripts"))
from dandi_data import load_pack

STATES  = ("Wake", "NREM", "REM")
COLORS  = {"Wake": "#D55E00", "NREM": "#0072B2", "REM": "#009E73"}
N_SUB   = 50          # matched population size; populations below this are excluded
DRAWS   = 50          # random subsamples per population and state
MIN_BINS = 300        # a state needs 5 minutes to give a spectrum
SEED    = 1
TABLE   = "multisession_50units_clauset.csv"
FIG_DIR = "figures"
os.makedirs(FIG_DIR, exist_ok=True)

mpl.rcParams.update({"font.size": 9, "axes.titlesize": 9, "axes.labelsize": 9,
                     "xtick.labelsize": 8, "ytick.labelsize": 8, "legend.fontsize": 7.5,
                     "figure.dpi": 110, "savefig.dpi": 300, "savefig.bbox": "tight"})

def population_rng(tag):
    """One deterministic stream per population, independent of iteration order."""
    h = int.from_bytes(hashlib.sha256(("pr50|" + tag).encode()).digest()[:4], "big")
    return np.random.default_rng(SEED ^ h)


In [ ]:
def spectrum(M):
    """Eigenvalues of the covariance of a bins x neurons matrix, as variance fractions.

    Mean-centred, NOT z-scored: the decomposition is of the covariance matrix,
    so a neuron with more variance contributes more to the leading components.
    """
    M = M - M.mean(0)
    ev = np.linalg.svd(M, compute_uv=False) ** 2
    return np.sort(ev / ev.sum())[::-1]


def participation_ratio(ev):
    return float((ev.sum() ** 2) / (ev ** 2).sum())


def d90(ev):
    """Number of leading components carrying 90 % of the variance."""
    return int(np.searchsorted(np.cumsum(ev), 0.90) + 1)


def clauset_fit(ev):
    """Maximum-likelihood power-law fit above a KS-selected cutoff (powerlaw pkg)."""
    f = powerlaw.Fit(ev, discrete=False, verbose=False).power_law
    return float(f.alpha), float(f.xmin), int(f.n), float(f.D)


In [ ]:
if os.path.exists(TABLE):
    P = pd.read_csv(TABLE)
else:
    pops = [p for p in load_pack(min_units=N_SUB)
            if all((p["state"] == s).sum() >= MIN_BINS for s in STATES)]
    rows = []
    for p in pops:
        rng = population_rng(f"{p['group']}|{p['tag']}")
        rec = dict(tag=p["tag"].split("_ses-")[-1], group=p["group"], n_units=p["X"].shape[1])
        ok = True
        for s in STATES:
            M = p["X"][p["state"] == s]
            live = np.flatnonzero(M.sum(0) >= 10)       # units with >= 10 spikes in this state
            if len(live) < N_SUB:
                ok = False
                break
            acc = {k: [] for k in ("pr", "d90", "alpha", "conv", "xmin", "n_tail", "D")}
            for _ in range(DRAWS):
                ev = spectrum(M[:, rng.choice(live, N_SUB, replace=False)])
                a, xm, nt, dks = clauset_fit(ev)
                acc["pr"].append(participation_ratio(ev)); acc["d90"].append(d90(ev))
                acc["alpha"].append(a); acc["conv"].append(1.0 / (a - 1.0))
                acc["xmin"].append(xm); acc["n_tail"].append(nt); acc["D"].append(dks)
            rec[f"D_pr_{s}"]       = float(np.mean(acc["pr"]))
            rec[f"D_90_{s}"]       = float(np.mean(acc["d90"]))
            rec[f"alpha_mle_{s}"]  = float(np.mean(acc["alpha"]))
            rec[f"alpha_conv_{s}"] = float(np.mean(acc["conv"]))
            rec[f"xmin_{s}"]       = float(np.mean(acc["xmin"]))
            rec[f"n_tail_{s}"]     = float(np.mean(acc["n_tail"]))
            rec[f"D_ks_{s}"]       = float(np.mean(acc["D"]))
            # sparseness is measured on the FIXED population, not the subsample
            rec[f"s_{s}"]          = float((M > 0).mean())
        if ok:
            rows.append(rec)
    P = pd.DataFrame(rows)
    P.round(4).to_csv(TABLE, index=False)

print(f"{len(P)} populations at the {N_SUB}-unit floor, {P.n_units.min()}-{P.n_units.max()} recorded units")
print(P.groupby("group").size().to_string())


In [ ]:
summary = pd.DataFrame({
    s: dict(D_pr=P[f"D_pr_{s}"].median(), D_90=P[f"D_90_{s}"].median(),
            alpha_mle=P[f"alpha_mle_{s}"].median(), alpha_conv=P[f"alpha_conv_{s}"].median(),
            n_tail=P[f"n_tail_{s}"].median(), D_ks=P[f"D_ks_{s}"].median(),
            active_fraction=P[f"s_{s}"].median())
    for s in STATES}).T.round(3)
print(summary.to_string())

print("\npaired comparisons across populations (Wilcoxon signed-rank):")
for col, name in (("D_pr", "participation ratio"), ("D_90", "components for 90 %"),
                  ("alpha_mle", "tail index")):
    for a, b in (("Wake", "NREM"), ("Wake", "REM"), ("NREM", "REM")):
        d = P[f"{col}_{a}"] - P[f"{col}_{b}"]
        print(f"  {name:22s} {a} > {b}: {int((d > 0).sum()):2d}/{len(d)}  "
              f"median diff {d.median():+7.3f}  p = {wilcoxon(d).pvalue:.1e}")


In [ ]:
long = pd.concat([P.assign(state=s, D_pr=P[f"D_pr_{s}"], D_90=P[f"D_90_{s}"],
                           a_conv=P[f"alpha_conv_{s}"], a_mle=P[f"alpha_mle_{s}"], s_act=P[f"s_{s}"])
                  [["tag", "group", "state", "D_pr", "D_90", "a_conv", "a_mle", "s_act"]]
                  for s in STATES], ignore_index=True)
print(len(long), "population-state observations")


In [ ]:
# ---- exponent by state, MAXIMUM LIKELIHOOD (replaces the least-squares panel)
fig, ax = plt.subplots(figsize=(4.2, 3.5), constrained_layout=True)
for i, s in enumerate(STATES):
    v = P[f"alpha_conv_{s}"].values
    q1, med, q3 = np.percentile(v, [25, 50, 75])
    ax.plot(np.full(len(v), i) + np.random.default_rng(i).uniform(-.12, .12, len(v)), v,
            "o", ms=2.8, mfc="none", mew=.6, color=COLORS[s], alpha=.55, zorder=1)
    ax.errorbar(i + 0.28, med, yerr=[[med - q1], [q3 - med]], fmt="o", ms=7.5,
                color=COLORS[s], elinewidth=1.6, capsize=4, zorder=3)
    ax.annotate(f"{med:.2f}", (i + 0.28, med), textcoords="offset points", xytext=(11, -3),
                fontsize=8, color=COLORS[s])
ax.set_xticks(range(3)); ax.set_xticklabels(STATES); ax.set_xlim(-0.45, 2.75)
ax.set_ylabel(r"exponent $\alpha = 1/(\hat{\alpha}-1)$   (maximum likelihood)")
ax.spines[["top", "right"]].set_visible(False)
fig.savefig(f"{FIG_DIR}/fig5_exponent_mle_by_state.png")
plt.show()


In [ ]:
# ---- the three relations
panels = [("fig5a_Dpr_vs_D90.png", "D_pr", "D_90",
           r"participation ratio  $D_{\mathrm{PR}}$", r"components for 90 % variance  $D_{90}$"),
          ("fig5b_alpha_vs_Dpr.png", "a_conv", "D_pr",
           r"tail index converted,  $1/(\hat{\alpha}-1)$", r"participation ratio  $D_{\mathrm{PR}}$"),
          ("fig5c_sparseness_vs_Dpr.png", "s_act", "D_pr",
           r"mean active fraction  $s$", r"participation ratio  $D_{\mathrm{PR}}$")]
for fname, xc, yc, xl, yl in panels:
    fig, ax = plt.subplots(figsize=(4.4, 3.6), constrained_layout=True)
    for s in STATES:
        d = long[long.state == s]
        r = spearmanr(d[xc], d[yc])
        ax.plot(d[xc], d[yc], "o", ms=5, mfc="none", mew=1.2, color=COLORS[s],
                label=f"{s}   $\\rho$ = {r.statistic:+.2f}" + ("" if r.pvalue < .05 else " (n.s.)"))
    rp = spearmanr(long[xc], long[yc])
    ax.set_xlabel(xl); ax.set_ylabel(yl)
    ax.legend(frameon=False, fontsize=7.5)
    ax.spines[["top", "right"]].set_visible(False)
    ax.margins(0.06)
    fig.savefig(f"{FIG_DIR}/{fname}")
    print(f"{fname}: pooled rho = {rp.statistic:+.3f} (p = {rp.pvalue:.1e})")
    plt.show()


## Is a power law an adequate description?

Estimating an exponent does not establish that the spectrum *is* a power law: on
log-log axes any smoothly decaying spectrum gives a high $R^2$. The test is
separate, and it is applied with the same convention as every other quantity
here - **computed on each subsample and then averaged within a population**,
never on an averaged spectrum.

For each 50-unit subsample the fit is made above a cutoff chosen by the
Kolmogorov-Smirnov criterion, and the observed KS distance $D$ is compared with
distances produced by synthetic power laws of the same size: each replicate draws
a fresh Pareto tail above the cutoff, resamples the body below it, and is refitted
from scratch including its own cutoff scan. The resulting $p$ is the fraction of
replicates fitting at least as badly, so it quantifies compatibility with the
power-law hypothesis rather than proving one, and it is conservative - the
eigenvalues are order statistics of one covariance matrix, not independent draws,
which makes the empirical curve smoother than the null.

$\hat{\alpha}$ and $D$ are averaged over all 50 draws (from the table computed
above); the bootstrap is the expensive part and is averaged over the first 10 of
the same draws, computed by `../scripts/gof50_perdraw_p.py`.

The panel shows the empirical distributions only. No fitted curve is drawn,
because with per-draw fitting there is no single fitted curve per state to draw -
the fit is 50 fits per population, and the numbers in the legend are their
average.


In [ ]:
PB = pd.read_csv("gof50_perdraw_p.csv")
S50 = np.load("gof50_spectra.npz")                 # mean matched spectrum per population, for display

gof_summary = pd.DataFrame({
    s: dict(alpha=P[f"alpha_mle_{s}"].median(), alpha_conv=P[f"alpha_conv_{s}"].median(),
            D=P[f"D_ks_{s}"].median(), n_tail=P[f"n_tail_{s}"].median(),
            p=PB[PB.state == s].p_boot.median(),
            not_rejected=float((PB[PB.state == s].p_boot >= .10).mean()),
            n_pop=int((PB.state == s).sum()))
    for s in STATES}).T.round(3)
print(gof_summary.to_string())

fig, ax = plt.subplots(figsize=(5.4, 4.0), constrained_layout=True)
for s in STATES:
    for _, r in PB[PB.state == s].iterrows():
        key = f"{r.group}|{r.tag}|{s}"
        if key not in S50:
            continue
        ev = S50[key]
        n = len(ev)
        ax.loglog(ev, np.arange(1, n + 1) / n, color=COLORS[s], lw=.7, alpha=.3, zorder=1)
    q = gof_summary.loc[s]
    ax.plot([], [], color=COLORS[s], lw=2.2,
            label=f"{s}   $\\hat{{\\alpha}}$ = {q.alpha:.2f} ($\\alpha$ = {q.alpha_conv:.2f}),  "
                  f"D = {q.D:.3f},  p = {q.p:.2f}")
ax.set_xlim(2e-3, 0.5); ax.set_ylim(1.5e-2, 1.5)
ax.set_xlabel("eigenvalue $\\lambda$  (fraction of total variance)")
ax.set_ylabel("fraction of eigenvalues $\\geq \\lambda$")
ax.legend(frameon=False, loc="lower left", fontsize=7)
ax.spines[["top", "right"]].set_visible(False)
fig.savefig(f"{FIG_DIR}/fig5_gof_cumulative_50units.png")
plt.show()


## First-order statistics

Everything above is a property of the population. These are properties of single
neurons, measured on the same bins: the mean firing rate, the Fano factor of the
1 s spike counts (variance over mean, a measure of how irregular a neuron's
count is from second to second) and the fraction of seconds in which a neuron
fires. They are reported because the population results have to be read against
them - a difference in dimensionality means little if it is simply a difference
in how fast the neurons fire.


In [ ]:
rows = []
keys = set(zip(P.tag, P.group))                      # the same 34 populations as above
for p in [q for q in load_pack(min_units=N_SUB)
          if (q["tag"].split("_ses-")[-1], q["group"]) in keys]:
    for s in STATES:
        M = p["X"][p["state"] == s].astype(float)
        mu, var = M.mean(0), M.var(0)
        keep = mu > 0
        rows.append(dict(tag=p["tag"].split("_ses-")[-1], group=p["group"], state=s,
                         rate=float(mu[keep].mean()), rate_median=float(np.median(mu[keep])),
                         fano=float(np.median(var[keep] / mu[keep])),
                         frac_bins_active=float((M > 0).mean())))
FO = pd.DataFrame(rows)
FO.round(4).to_csv("first_order_stats.csv", index=False)
print(FO.groupby("state")[["rate", "rate_median", "fano", "frac_bins_active"]]
      .median().reindex(list(STATES)).round(3).to_string())
print("\npaired across populations:")
for a, b in (("Wake", "NREM"), ("NREM", "REM"), ("Wake", "REM")):
    out = []
    for col in ("rate", "fano", "frac_bins_active"):
        d = (FO[FO.state == a].set_index(["tag", "group"])[col]
             - FO[FO.state == b].set_index(["tag", "group"])[col]).dropna()
        out.append(f"{col} {int((d > 0).sum())}/{len(d)} p={wilcoxon(d).pvalue:.1e}")
    print(f"  {a} vs {b}: " + " | ".join(out))


## ON and OFF states

The 1 s active fraction shows that non-REM alternates between near-silence and
widespread activity, but a 1 s bin cannot resolve the alternation itself. At 10
ms the population is either firing (ON) or completely silent (OFF). An OFF state
is defined here as at least 50 ms in which none of the retained neurons fires.

Two cautions make the control necessary. With this many neurons at these rates,
short empty windows occur by chance even without any coordination, so the
observed OFF fraction is compared with a surrogate in which each neuron keeps its
spike count within each bout but its spikes are placed at random times -
destroying alignment between neurons while preserving every rate. And this
analysis needs millisecond resolution, so it reads the original spike times of
the reference recording rather than the 1 s pack; it is therefore a
single-recording result.


In [ ]:
import pickle
PKL   = "../data/spike_time_dicts/buzsaki_Achilles_10252013.pkl"
PACK  = "../data/dandi_binned_1s_rat.npz"
TAG   = "000044_sub-Achilles_ses-Achilles-10252013"
THRESH, FINE, OFF_MIN = 4.71, 0.010, 0.050

zz   = np.load(PACK, allow_pickle=True)
lab  = zz[f"state__{TAG}"].copy(); lab[lab == "Quiet"] = "Wake"
dur  = zz[f"X__{TAG}"].shape[0] * 1.0
dpkl = pickle.load(open(PKL, "rb"))
keep_units = [k for k, v in dpkl.items() if len(np.asarray(v)) / dur <= THRESH]
spk  = np.sort(np.concatenate([np.asarray(dpkl[k], float) for k in keep_units]))

def bouts_of(state, min_len=20):
    m = (lab == state).astype(int)
    d = np.diff(np.concatenate([[0], m, [0]]))
    return [(float(a), float(b)) for a, b in zip(np.flatnonzero(d == 1), np.flatnonzero(d == -1))
            if b - a >= min_len]

def off_stats(times, a, b):
    edges = np.arange(a, b, FINE)
    c, _ = np.histogram(times, bins=edges)
    silent = c == 0
    d = np.diff(np.concatenate([[0], silent.astype(int), [0]]))
    lens = (np.flatnonzero(d == -1) - np.flatnonzero(d == 1)) * FINE
    ok = lens >= OFF_MIN
    return float(lens[ok].sum()), int(ok.sum()), (len(edges) - 1) * FINE, lens[ok]

rng_s, rows = np.random.default_rng(SEED), []
for s in STATES:
    bb = bouts_of(s)
    off_t = tot = 0.0; off_n = 0; durs = []; sur_t = 0.0
    for a, b in bb:
        if b - a < 1:
            continue
        t_, n_, T_, L_ = off_stats(spk[(spk >= a) & (spk < b)], a, b)
        off_t += t_; off_n += n_; tot += T_; durs += L_.tolist()
        counts = [int(((np.asarray(dpkl[k], float) >= a) & (np.asarray(dpkl[k], float) < b)).sum())
                  for k in keep_units]
        fake = np.sort(np.concatenate([rng_s.uniform(a, b, n) for n in counts if n])) if any(counts) else np.array([])
        sur_t += off_stats(fake, a, b)[0]
    rows.append(dict(state=s, bouts=len(bb), seconds=round(tot),
                     off_fraction=round(off_t / tot, 4), surrogate_off=round(sur_t / tot, 4),
                     ratio=round((off_t / tot) / (sur_t / tot), 2),
                     off_per_min=round(60 * off_n / tot, 1),
                     median_off_ms=round(1000 * float(np.median(durs)), 1)))
ONOFF = pd.DataFrame(rows)
ONOFF.to_csv("onoff_states_achilles.csv", index=False)
print(ONOFF.to_string(index=False))

from matplotlib.patches import Patch
fig, ax = plt.subplots(figsize=(4.6, 3.5), constrained_layout=True)
for i, s in enumerate(STATES):
    r = ONOFF[ONOFF.state == s].iloc[0]
    ax.bar(i - 0.17, r.off_fraction, width=0.32, color=COLORS[s])
    ax.bar(i + 0.17, r.surrogate_off, width=0.32, color=COLORS[s], alpha=.32,
           hatch="///", edgecolor=COLORS[s])
    ax.annotate(f"{r.off_fraction:.2f}", (i - 0.17, r.off_fraction), ha="center",
                textcoords="offset points", xytext=(0, 3), fontsize=7.5, color=COLORS[s])
ax.set_xticks(range(3)); ax.set_xticklabels(STATES)
ax.set_ylabel("fraction of time in population OFF states")
ax.legend(handles=[Patch(facecolor="0.45", label="observed"),
                   Patch(facecolor="0.85", hatch="///", edgecolor="0.45",
                         label="rate-matched surrogate")],
          frameon=False, fontsize=7.5, loc="upper right")
ax.spines[["top", "right"]].set_visible(False)
ax.margins(y=0.14)
fig.savefig(f"{FIG_DIR}/fig3f_off_states.png")
plt.show()


## What the panels show

**(a)** The two range-free measures agree, so $D_{\mathrm{PR}}$ and $D_{90}$ are
reading the same property of the spectrum.

**(b)** The tail index is only loosely related to either of them, and within
single states the relation is weak. The exponent describes how the upper tail
decays; it is not a substitute for a dimensionality measure.

**(c)** Populations that recruit more neurons per second are
higher-dimensional, driven by the waking points. Note the sign: $\hat{\alpha}$
measures steepness, so a positive relation here and the negative relation
against the exponent in the single-session chapter are the same finding.

Caveat for any regional statement: the 50-neuron floor keeps the larger
populations, which are predominantly amygdala. Every comparison above is paired
within a population, so the floor does not bias the state contrasts, but the
regional composition is not representative of the archive.
